Cresta uses "Synthetic Customers" derived from real conversation data to stress-test AI agents. How would you generate realistic synthetic test personas from a dataset of 50,000 transcripts? What makes a synthetic persona "representative"?

1. Extract structured features from each transcript.

Run the 50,000 transcripts through an LLM (or NLP pipeline) that tags each conversation with attributes you care about:

Intent — why they contacted you (billing dispute, cancellation, technical issue, refund)
Emotional tone — calm, frustrated, confused, angry
Communication style — terse, rambling, formal, types in fragments
Behavioral traits — interrupts, repeats themselves, goes off-topic, demands a human
Outcome — resolved, escalated, abandoned

2. Cluster the conversations.

Embed those features (or the raw transcripts) into vectors and run clustering (k-means, HDBSCAN). Each cluster is a naturally occurring "type" of customer — e.g. "frustrated cancel-er who already tried self-service," "confused first-timer asking basic questions."

3. Turn clusters into persona templates.

For each meaningful cluster, summarize it into a persona spec: goal, mood, style, knowledge level, and likely curveballs. Keep the distribution — how big each cluster is — because that's what makes the test set realistic.

4. Generate synthetic personas from the templates.

Use an LLM to instantiate variations within each cluster (different phrasings, names, edge details) so you get diversity without inventing customer types that never appear in the real data. The persona then "plays" the role against the AI agent, improvising like a real person would.

What makes a persona "representative"
Three things, in order of importance:
Distributional fidelity. The mix of personas should match the real population. If 30% of real contacts are frustrated billing disputes, ~30% of your test personas should be too. A test set that's all easy or all hard tells you nothing about real-world performance.
Behavioral realism, not just topical. A representative persona doesn't just say the right thing — it behaves like the cluster it came from: gets impatient, misunderstands, changes its mind, withholds information until asked. The hard part of stress-testing is the messy interaction, not the topic.
Coverage of the tails. Beyond the common cases, you deliberately include the rare-but-costly ones (the angry escalator, the ambiguous edge case, the adversarial user). These are underrepresented by frequency but matter disproportionately for safety and failure detection.



In [ ]:
"""
Synthetic Customer Persona Generation Pipeline
==============================================

Turns a corpus of real support transcripts into a *representative* set of
synthetic test personas for stress-testing an AI agent.

Pipeline:
    transcripts -> feature extraction -> embed -> cluster
                -> persona templates (with real-world weights)
                -> instantiate synthetic personas
                -> representativeness report

Two modes:
    USE_LLM=False (default)  -> runs fully offline using TF-IDF + KMeans +
                                a deterministic mock extractor. Good for a demo.
    USE_LLM=True             -> swaps in real LLM calls + embedding model.
                                See the two clearly-marked functions below.

Run:  python synthetic_personas.py
"""

from __future__ import annotations

import json
import random
from collections import Counter
from dataclasses import dataclass, asdict, field
from typing import List, Dict

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
USE_LLM = False          # flip to True in production (see _llm_* functions)
N_CLUSTERS = 6           # in prod, pick via silhouette score or use HDBSCAN
PERSONAS_PER_1000 = 1    # sampling rate -> total personas = N * rate
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ---------------------------------------------------------------------------
# Data structures
# ---------------------------------------------------------------------------
@dataclass
class Transcript:
    id: str
    text: str


@dataclass
class Features:
    """Structured attributes extracted from one transcript."""
    intent: str            # billing | cancel | tech_issue | refund | how_to
    emotion: str           # calm | frustrated | confused | angry
    style: str             # terse | rambling | formal
    self_service_tried: bool
    outcome: str           # resolved | escalated | abandoned


@dataclass
class PersonaTemplate:
    """A cluster summarized into a reusable persona spec."""
    cluster_id: int
    label: str
    weight: float                      # share of the real population [0,1]
    dominant_intent: str
    dominant_emotion: str
    dominant_style: str
    escalation_rate: float
    curveballs: List[str] = field(default_factory=list)


@dataclass
class SyntheticPersona:
    persona_id: str
    template_label: str
    intent: str
    emotion: str
    style: str
    backstory: str
    system_prompt: str                 # what you feed the role-playing LLM


# ---------------------------------------------------------------------------
# Step 1: Feature extraction
# ---------------------------------------------------------------------------
def extract_features(t: Transcript) -> Features:
    if USE_LLM:
        return _llm_extract_features(t)
    return _mock_extract_features(t)


def _llm_extract_features(t: Transcript) -> Features:
    """PRODUCTION: ask an LLM to tag the transcript with a strict JSON schema."""
    import anthropic
    client = anthropic.Anthropic()
    prompt = (
        "Tag this support transcript. Respond ONLY with JSON matching:\n"
        '{"intent": "billing|cancel|tech_issue|refund|how_to", '
        '"emotion": "calm|frustrated|confused|angry", '
        '"style": "terse|rambling|formal", '
        '"self_service_tried": true/false, '
        '"outcome": "resolved|escalated|abandoned"}\n\n'
        f"TRANSCRIPT:\n{t.text}"
    )
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = "".join(b.text for b in msg.content if b.type == "text")
    data = json.loads(raw.replace("```json", "").replace("```", "").strip())
    return Features(**data)


def _mock_extract_features(t: Transcript) -> Features:
    """OFFLINE: cheap keyword heuristics so the demo runs without an API."""
    txt = t.text.lower()
    intent = next((k for k, kws in {
        "billing":   ["charge", "bill", "invoice", "payment"],
        "cancel":    ["cancel", "unsubscribe", "close my account"],
        "refund":    ["refund", "money back", "reimburse"],
        "tech_issue":["error", "broken", "not working", "crash", "bug"],
        "how_to":    ["how do i", "how to", "where is", "set up"],
    }.items() if any(w in txt for w in kws)), "how_to")

    emotion = ("angry" if any(w in txt for w in ["ridiculous", "unacceptable", "furious"])
               else "frustrated" if any(w in txt for w in ["still", "again", "third time"])
               else "confused" if "?" in txt and "how" in txt
               else "calm")

    style = ("rambling" if len(txt.split()) > 40
             else "formal" if any(w in txt for w in ["dear", "regards", "kindly"])
             else "terse")

    self_service_tried = any(w in txt for w in ["already tried", "the website", "faq", "help page"])
    outcome = ("escalated" if emotion in ("angry",) or "manager" in txt
               else "abandoned" if "never mind" in txt
               else "resolved")
    return Features(intent, emotion, style, self_service_tried, outcome)


# ---------------------------------------------------------------------------
# Step 2: Embed + Step 3: Cluster
# ---------------------------------------------------------------------------
def embed(transcripts: List[Transcript]) -> np.ndarray:
    if USE_LLM:
        return _llm_embed(transcripts)
    # OFFLINE: TF-IDF is a fine stand-in for sentence embeddings in a demo.
    vec = TfidfVectorizer(max_features=512, stop_words="english")
    return vec.fit_transform([t.text for t in transcripts]).toarray()


def _llm_embed(transcripts: List[Transcript]) -> np.ndarray:
    """PRODUCTION: real sentence embeddings."""
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model.encode([t.text for t in transcripts], show_progress_bar=True)


def cluster(embeddings: np.ndarray, k: int = N_CLUSTERS) -> np.ndarray:
    # In prod prefer HDBSCAN (no fixed k) or pick k by silhouette score.
    return KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10).fit_predict(embeddings)


# ---------------------------------------------------------------------------
# Step 3b: Build persona templates from clusters (keeping real-world weights)
# ---------------------------------------------------------------------------
def build_templates(features: List[Features], labels: np.ndarray) -> List[PersonaTemplate]:
    templates: List[PersonaTemplate] = []
    total = len(features)
    for cid in sorted(set(labels)):
        members = [f for f, lab in zip(features, labels) if lab == cid]
        if not members:
            continue
        intent = Counter(f.intent for f in members).most_common(1)[0][0]
        emotion = Counter(f.emotion for f in members).most_common(1)[0][0]
        style = Counter(f.style for f in members).most_common(1)[0][0]
        esc = sum(f.outcome == "escalated" for f in members) / len(members)
        tried = sum(f.self_service_tried for f in members) / len(members)

        curveballs = []
        if tried > 0.5:
            curveballs.append("already attempted self-service; don't repeat the FAQ")
        if emotion in ("angry", "frustrated"):
            curveballs.append("demands a human if first reply feels scripted")
        if style == "rambling":
            curveballs.append("buries the actual question inside a long story")

        templates.append(PersonaTemplate(
            cluster_id=cid,
            label=f"{emotion}_{intent}",
            weight=round(len(members) / total, 4),
            dominant_intent=intent,
            dominant_emotion=emotion,
            dominant_style=style,
            escalation_rate=round(esc, 3),
            curveballs=curveballs,
        ))
    return templates


# ---------------------------------------------------------------------------
# Step 4: Instantiate synthetic personas (weighted by real distribution)
# ---------------------------------------------------------------------------
NAMES = ["Alex", "Sam", "Priya", "Diego", "Mei", "Tom", "Fatima", "Jordan"]


def instantiate_personas(templates: List[PersonaTemplate], n_personas: int) -> List[SyntheticPersona]:
    weights = [t.weight for t in templates]
    chosen = random.choices(templates, weights=weights, k=n_personas)
    personas: List[SyntheticPersona] = []
    for i, tpl in enumerate(chosen):
        name = random.choice(NAMES)
        backstory = _backstory(tpl, name)
        sys_prompt = _role_play_prompt(tpl, name, backstory)
        personas.append(SyntheticPersona(
            persona_id=f"persona_{i:04d}",
            template_label=tpl.label,
            intent=tpl.dominant_intent,
            emotion=tpl.dominant_emotion,
            style=tpl.dominant_style,
            backstory=backstory,
            system_prompt=sys_prompt,
        ))
    return personas


def _backstory(tpl: PersonaTemplate, name: str) -> str:
    detail = {
        "billing": "was charged twice this month",
        "cancel": "wants to cancel before the next renewal",
        "refund": "returned an item two weeks ago and saw no refund",
        "tech_issue": "the app keeps crashing on login",
        "how_to": "can't find where to update payment info",
    }.get(tpl.dominant_intent, "has a general question")
    return f"{name} {detail}. Mood: {tpl.dominant_emotion}. Speaks in a {tpl.dominant_style} way."


def _role_play_prompt(tpl: PersonaTemplate, name: str, backstory: str) -> str:
    cb = "\n".join(f"- {c}" for c in tpl.curveballs) or "- none"
    return (
        f"You are role-playing a customer named {name}, NOT an assistant.\n"
        f"Situation: {backstory}\n"
        f"Stay in character. Be {tpl.dominant_emotion} and {tpl.dominant_style}.\n"
        f"Behaviors:\n{cb}\n"
        f"Only reveal details when asked. Reply with one customer message at a time."
    )


# ---------------------------------------------------------------------------
# Representativeness report
# ---------------------------------------------------------------------------
def representativeness_report(real: List[Features],
                             personas: List[SyntheticPersona]) -> Dict:
    def dist(counter: Counter, total: int) -> Dict[str, float]:
        return {k: round(v / total, 3) for k, v in counter.items()}

    real_intent = dist(Counter(f.intent for f in real), len(real))
    syn_intent = dist(Counter(p.intent for p in personas), len(personas))

    # L1 distance between the two intent distributions (0 = perfect match).
    keys = set(real_intent) | set(syn_intent)
    drift = round(sum(abs(real_intent.get(k, 0) - syn_intent.get(k, 0)) for k in keys), 3)

    return {
        "real_intent_distribution": real_intent,
        "synthetic_intent_distribution": syn_intent,
        "distribution_drift_L1": drift,   # lower is better; <0.1 is good
        "n_real": len(real),
        "n_personas": len(personas),
    }


# ---------------------------------------------------------------------------
# Demo data + main
# ---------------------------------------------------------------------------
def make_demo_corpus(n: int = 5000) -> List[Transcript]:
    seeds = [
        "I was charged twice on my bill this month, please fix the payment.",
        "I want to cancel my subscription before it renews, I already tried the website.",
        "The app keeps crashing on login, this is the third time, unacceptable.",
        "How do I update my payment info? I can't find it, where is it?",
        "I returned the item two weeks ago and still no refund, getting frustrated.",
        "Dear team, kindly advise how to set up two-factor authentication. Regards.",
    ]
    out = []
    for i in range(n):
        base = random.choice(seeds)
        # add a little noise/variation
        extra = random.choice(["", " Please help.", " This is urgent.",
                               " I've been a customer for years.", " Never mind, forget it."])
        out.append(Transcript(id=f"t{i}", text=base + extra))
    return out


def main():
    print(f"USE_LLM = {USE_LLM}\n")

    transcripts = make_demo_corpus(5000)
    print(f"[1] Loaded {len(transcripts)} transcripts")

    features = [extract_features(t) for t in transcripts]
    print(f"[1] Extracted features. Sample: {asdict(features[0])}")

    emb = embed(transcripts)
    labels = cluster(emb)
    print(f"[2/3] Embedded -> {emb.shape}, clustered into {len(set(labels))} groups")

    templates = build_templates(features, labels)
    print("\n[3] Persona templates (label | weight | escalation):")
    for t in sorted(templates, key=lambda x: -x.weight):
        print(f"    {t.label:24s} w={t.weight:<6} esc={t.escalation_rate}")

    n_personas = max(10, len(transcripts) * PERSONAS_PER_1000 // 1000)
    personas = instantiate_personas(templates, n_personas)
    print(f"\n[4] Generated {len(personas)} synthetic personas")
    print("    Example persona system prompt:\n")
    print("    " + personas[0].system_prompt.replace("\n", "\n    "))

    print("\n[5] Representativeness report:")
    report = representativeness_report(features, personas)
    print(json.dumps(report, indent=2))

    with open("synthetic_personas_output.json", "w") as f:
        json.dump([asdict(p) for p in personas], f, indent=2)
    print("\nWrote personas -> synthetic_personas_output.json")


if __name__ == "__main__":
    main()

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "I love machine learning",
    "I love Python",
    "Python is great for machine learning"
]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(docs)

print(vectorizer.get_feature_names_out())
print(X.toarray())
print(X)

['for' 'great' 'is' 'learning' 'love' 'machine' 'python']
[[0.         0.         0.         0.57735027 0.57735027 0.57735027
  0.        ]
 [0.         0.         0.         0.         0.70710678 0.
  0.70710678]
 [0.45954803 0.45954803 0.45954803 0.34949812 0.         0.34949812
  0.34949812]]
  (0, 3)	0.5773502691896257
  (0, 5)	0.5773502691896257
  (0, 4)	0.5773502691896257
  (1, 6)	0.7071067811865476
  (1, 4)	0.7071067811865476
  (2, 0)	0.45954803293870056
  (2, 1)	0.45954803293870056
  (2, 2)	0.45954803293870056
  (2, 6)	0.3494981241087058
  (2, 3)	0.3494981241087058
  (2, 5)	0.3494981241087058


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

X = vectorizer.fit_transform(docs)

similarity = cosine_similarity(X)
similarity

array([[1.        , 0.40824829, 0.40356567],
       [0.40824829, 1.        , 0.24713249],
       [0.40356567, 0.24713249, 1.        ]])